# 🤖 LangChain + ML Pipeline — Beginner's Guide

**What you'll build:** A complete AI pipeline that:
1. Loads & cleans data
2. Uses a free LLM (Google Gemini) via LangChain
3. Builds prompt templates
4. Chains steps together
5. Adds memory to conversations
6. Builds a simple RAG (Retrieval-Augmented Generation) system

**Free tools used:**
- 🟣 [Google Gemini API](https://aistudio.google.com/app/apikey) — free tier (get key in 30 sec)
- 🦜 LangChain — open source
- 🤗 HuggingFace embeddings — free, runs locally
- 🗄️ ChromaDB — free local vector store
- 📓 Google Colab — free GPU/CPU

---
> **Before you start:** Get a free Gemini API key at https://aistudio.google.com/app/apikey  
> Then paste it when prompted below.

## 📦 Step 1 — Install Dependencies

Run this once. It installs LangChain, Gemini integration, HuggingFace, and ChromaDB.

In [ ]:
%%capture
!pip install langchain langchain-google-genai langchain-community \
             chromadb sentence-transformers \
             google-generativeai tiktoken

print("✅ All packages installed!")

## 🔑 Step 2 — Set Up Your API Key

We use Colab's `userdata` to keep your key safe (not printed in output).

In [ ]:
import os
from getpass import getpass

# Option A: type it in securely
GOOGLE_API_KEY = getpass("🔑 Paste your Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("✅ API key set! (hidden for security)")

## 🧠 Step 3 — Understand the Core Concepts

Before writing chains, let's understand the **3 building blocks** of LangChain:

```
┌──────────────────────────────────────────────────────┐
│  LangChain Pipeline                                  │
│                                                      │
│  [Input] → [Prompt Template] → [LLM] → [Output]     │
│                                                      │
│  You can chain these steps together like LEGO blocks │
└──────────────────────────────────────────────────────┘
```

| Concept | What it does | Analogy |
|---|---|---|
| **LLM** | The AI brain (Gemini here) | A smart employee |
| **Prompt Template** | Structures your input | A form/template |
| **Chain** | Connects steps together | An assembly line |
| **Memory** | Remembers past messages | Short-term memory |
| **Retriever** | Fetches relevant docs | A smart search engine |
| **Vector Store** | Stores text as numbers | A searchable library |

## 🤖 Step 4 — Your First LLM Call

The simplest possible LangChain usage — just talk to the model.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# ── Create the LLM object ──────────────────────────────
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",   # free & fast
    temperature=0.7,             # 0 = deterministic, 1 = creative
)

# ── Send a message ────────────────────────────────────
response = llm.invoke("What is machine learning in one sentence?")

print("📤 Question: What is machine learning in one sentence?")
print("📥 Answer:", response.content)

## 📝 Step 5 — Prompt Templates

Hard-coding questions is inflexible. **Prompt Templates** let you define a reusable structure with variables.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ── Define a template with {variables} ────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful teacher who explains {subject} simply."),
    ("human",  "Explain {topic} in 2-3 sentences for a beginner.")
])

# ── Fill in the variables ──────────────────────────────
filled_prompt = prompt.format_messages(
    subject="data science",
    topic="neural networks"
)

print("📋 Filled Prompt:")
for msg in filled_prompt:
    print(f"  [{msg.type.upper()}]: {msg.content}")

# ── Send it to the LLM ────────────────────────────────
response = llm.invoke(filled_prompt)
print("\n📥 Response:", response.content)

## ⛓️ Step 6 — Building a Chain (LCEL)

**LCEL** (LangChain Expression Language) uses the `|` pipe operator to chain steps — just like Unix pipes.

```
prompt | llm | output_parser
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Each piece ────────────────────────────────────────
prompt = ChatPromptTemplate.from_template(
    "Give me 3 real-world applications of {concept} in ML. Be concise."
)

output_parser = StrOutputParser()   # converts LLM output → plain string

# ── Chain them with | ─────────────────────────────────
#   Input dict → prompt → llm → string
chain = prompt | llm | output_parser

# ── Run the chain ─────────────────────────────────────
result = chain.invoke({"concept": "decision trees"})
print("📥 Result:", result)

print("\n" + "─"*50)
print("💡 Try changing the concept below:")

# Reuse the same chain with a different input!
result2 = chain.invoke({"concept": "k-means clustering"})
print("📥 Result 2:", result2)

## 🔁 Step 7 — Adding Memory (Conversation History)

Without memory, the LLM forgets every message. Let's fix that.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# ── A prompt that includes chat history ───────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful ML tutor. Keep answers brief."),
    MessagesPlaceholder(variable_name="history"),   # ← history goes here
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# ── Manually maintain history (simple approach) ───────
history = []

def chat(question):
    response = chain.invoke({"history": history, "question": question})
    # Append both turns to history
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=response))
    return response

# ── Multi-turn conversation ────────────────────────────
print("Turn 1:")
print(chat("What is overfitting in ML?"))

print("\nTurn 2 (references Turn 1):")
print(chat("How do I prevent what you just described?"))  # 'what you just described' = overfitting

print("\nTurn 3:")
print(chat("Give me one concrete example of the first technique you mentioned."))

## 📚 Step 8 — RAG (Retrieval-Augmented Generation)

RAG lets your LLM **answer questions about your own documents**.

```
┌─────────────────────────────────────────────────────────┐
│  RAG Pipeline                                           │
│                                                         │
│  Your Docs → Embeddings → Vector Store                  │
│                                ↓                        │
│  User Question → Retrieve Relevant Chunks → LLM → Answer│
└─────────────────────────────────────────────────────────┘
```

**Step 8a** — Create sample documents

In [ ]:
from langchain_core.documents import Document

# Our "knowledge base" — replace with your own content!
documents = [
    Document(
        page_content="Random Forest is an ensemble method that builds many decision trees and "
                     "combines their predictions. It reduces overfitting compared to a single tree "
                     "and is robust to outliers. Feature importance can be derived from it.",
        metadata={"source": "ml_notes", "topic": "random_forest"}
    ),
    Document(
        page_content="Gradient Boosting builds trees sequentially, where each tree corrects the "
                     "errors of the previous one. XGBoost and LightGBM are popular implementations. "
                     "It often wins Kaggle tabular competitions.",
        metadata={"source": "ml_notes", "topic": "gradient_boosting"}
    ),
    Document(
        page_content="Support Vector Machines find the hyperplane that best separates classes with "
                     "maximum margin. The kernel trick allows SVMs to handle non-linear data. "
                     "They work well in high-dimensional spaces but are slow on large datasets.",
        metadata={"source": "ml_notes", "topic": "svm"}
    ),
    Document(
        page_content="Logistic Regression is a simple, interpretable linear model for classification. "
                     "Despite its name, it is a classification algorithm. Output probabilities via "
                     "the sigmoid function. Good baseline model.",
        metadata={"source": "ml_notes", "topic": "logistic_regression"}
    ),
    Document(
        page_content="K-Nearest Neighbors (KNN) classifies a point by majority vote of its k closest "
                     "training samples. No training phase — it memorizes data. Sensitive to feature "
                     "scaling and curse of dimensionality.",
        metadata={"source": "ml_notes", "topic": "knn"}
    ),
]

print(f"📚 Loaded {len(documents)} documents into memory")
print(f"📄 Sample:\n  '{documents[0].page_content[:80]}...'")

**Step 8b** — Create embeddings and store in ChromaDB

> Embeddings convert text → numbers so we can measure similarity. HuggingFace runs this **locally for free**.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("⏳ Loading embedding model (downloads ~90MB once)...")

# Free local embedding model — no API key needed!
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",   # small, fast, good quality
    model_kwargs={"device": "cpu"}
)

# Create vector store from our documents
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ml_knowledge_base"
)

print(f"✅ Vector store created with {vectorstore._collection.count()} documents")

**Step 8c** — Test semantic search

In [ ]:
# Search for relevant documents
query = "Which algorithm is good for competitions and corrects errors?"
results = vectorstore.similarity_search(query, k=2)

print(f"🔍 Query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} [topic: {doc.metadata['topic']}]")
    print(f"  {doc.page_content}\n")

**Step 8d** — Build the full RAG chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Retriever: fetches top-k relevant docs ─────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ── Prompt that includes retrieved context ─────────────
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the context doesn't contain the answer, say "I don't know."

Context:
{context}

Question: {question}

Answer:
""")

# ── Helper to format docs into a string ───────────────
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# ── Full RAG chain ────────────────────────────────────
rag_chain = (
    {"context": retriever | format_docs,   # retrieve + format
     "question": RunnablePassthrough()}    # pass question through
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Ask questions about your documents! ───────────────
questions = [
    "What is the main advantage of Random Forest over a single decision tree?",
    "Which algorithms work well in high-dimensional spaces?",
    "What is the capital of France?",   # not in our docs → should say I don't know
]

for q in questions:
    print(f"❓ {q}")
    print(f"💬 {rag_chain.invoke(q)}\n")

## 🔀 Step 9 — Sequential Chain (Multi-Step Pipeline)

Chain multiple LLM calls together — output of one becomes input of the next.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Step A: Explain an ML concept ─────────────────────
explain_prompt = ChatPromptTemplate.from_template(
    "Explain the ML concept '{concept}' in 2 sentences."
)

# ── Step B: Generate a Python code skeleton ────────────
code_prompt = ChatPromptTemplate.from_template(
    """Based on this explanation: {explanation}
    
Write a minimal Python sklearn code example. Include comments. Keep it under 20 lines."""
)

# ── Step C: Suggest improvements ──────────────────────
improve_prompt = ChatPromptTemplate.from_template(
    """Given this code: {code}
    
List 2 specific improvements a beginner could make to it."""
)

parser = StrOutputParser()

# ── Chain them sequentially ───────────────────────────
explain_chain = explain_prompt | llm | parser

full_pipeline = (
    {"explanation": explain_chain, "concept": RunnablePassthrough()}
    | code_prompt | llm | parser
)

# ── Run the pipeline ──────────────────────────────────
concept = "logistic regression"
print(f"🔄 Running 2-step pipeline for: '{concept}'\n")

explanation = explain_chain.invoke({"concept": concept})
print("📖 Explanation:")
print(explanation)

print("\n💻 Generated Code:")
code = (code_prompt | llm | parser).invoke({"explanation": explanation})
print(code)

print("\n🔧 Suggested Improvements:")
improvements = (improve_prompt | llm | parser).invoke({"code": code})
print(improvements)

## 🗺️ Step 10 — Full Pipeline Summary

Here's everything you learned visualized:

In [ ]:
pipeline_summary = """
╔══════════════════════════════════════════════════════════════╗
║          LANGCHAIN PIPELINE — WHAT YOU BUILT                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  [Step 4]  Direct LLM Call                                   ║
║            User Input ──→ LLM ──→ Response                   ║
║                                                              ║
║  [Step 5]  Prompt Templates                                  ║
║            Variables ──→ Template ──→ LLM ──→ Response       ║
║                                                              ║
║  [Step 6]  LCEL Chain (|)                                    ║
║            Input ──→ Prompt | LLM | Parser ──→ String        ║
║                                                              ║
║  [Step 7]  Memory                                            ║
║            Question + History ──→ LLM ──→ Answer             ║
║            (stores turns so LLM remembers context)           ║
║                                                              ║
║  [Step 8]  RAG                                               ║
║            Docs ──→ Embeddings ──→ Vector Store              ║
║            Question ──→ Retrieve ──→ Augment ──→ LLM ──→ Ans ║
║                                                              ║
║  [Step 9]  Sequential Chain                                  ║
║            Input ──→ Chain A ──→ Chain B ──→ Chain C ──→ Out ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(pipeline_summary)

## 🚀 Step 11 — Mini Project: ML Study Assistant

Combine everything into a single interactive ML study assistant!

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

class MLStudyAssistant:
    """A simple ML tutor that remembers the conversation."""
    
    def __init__(self, llm, vectorstore):
        self.llm = llm
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
        self.history = []
        
        self.prompt = ChatPromptTemplate.from_messages([
            ("system",
             """You are an ML tutor. Use the context below if relevant, """
             """otherwise use your own knowledge. Be concise and friendly.\n"""
             """\nContext from knowledge base:\n{context}"""),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}")
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()
    
    def ask(self, question):
        # Retrieve relevant context
        docs = self.retriever.invoke(question)
        context = "\n".join(d.page_content for d in docs)
        
        # Get answer
        answer = self.chain.invoke({
            "context": context,
            "history": self.history,
            "question": question
        })
        
        # Update history
        self.history.append(HumanMessage(content=question))
        self.history.append(AIMessage(content=answer))
        
        return answer
    
    def reset(self):
        self.history = []
        print("🔄 Conversation reset.")


# ── Create and use the assistant ──────────────────────
from langchain_core.prompts import MessagesPlaceholder

assistant = MLStudyAssistant(llm, vectorstore)

# Simulated study session
session = [
    "What ML algorithm should I start learning first as a beginner?",
    "Why did you suggest that one specifically?",
    "How does it compare to the algorithm that wins Kaggle competitions?",
]

print("📚 ML Study Session\n" + "="*50)
for question in session:
    print(f"\n🙋 You: {question}")
    print(f"🤖 Tutor: {assistant.ask(question)}")

## 🎯 What's Next?

| Concept | Learn Next | Resource |
|---|---|---|
| **Agents** | LLMs that call tools & take actions | LangChain Agents docs |
| **LangSmith** | Debug & trace your chains | smith.langchain.com (free tier) |
| **Streaming** | Stream responses word-by-word | `chain.stream(...)` |
| **Custom Tools** | Give LLM tools like calculators | `@tool` decorator |
| **LangGraph** | Stateful multi-agent workflows | LangGraph docs |
| **Fine-tuning** | Customize a model on your data | HuggingFace PEFT |

---

### 🔑 Key Takeaways
1. **LangChain = glue** between your code, prompts, and LLMs
2. **Chains** (`|`) let you build pipelines like assembly lines
3. **RAG** = give LLMs knowledge from your own documents
4. **Memory** = manually pass history to keep context
5. **Everything is composable** — mix and match as needed

> 💡 **Experiment tip:** Change the `documents` in Step 8 to your own notes, PDFs, or articles — you've built your own private AI tutor!